# Alpha-synuclein/dopamine Kaggle demo

Environment and integration demo only. This notebook does not run the guarded 15-job scientific batch.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

PROJECT_URL = 'https://github.com/TanVi3001/drosophila-pd-neural-disease.git'
PROJECT_REF = 'review/gate-20e-promote-parkin-class-level-signoff'
FLYGIM_URL = 'https://github.com/TanVi3001/drosophila-pd-flygym.git'
FLYGIM_REF = 'main'
BRAIN_DATASET_SLUG = 'drosophila-pd-fly-brain-v1'  # change to your attached Dataset slug
DEMO_STEPS = 100
RUN_DEMO = False

WORK = Path('/kaggle/working')
PROJECT = WORK / 'drosophila-pd-neural-disease'
FLYGIM = WORK / 'drosophila-pd-flygym'
BRAIN_INPUT = Path('/kaggle/input') / BRAIN_DATASET_SLUG
BRAIN = PROJECT / 'external' / 'fly-brain'
DEMO_OUTPUT = WORK / 'kaggle_alpha_syn_dopamine_demo'
print({'python': sys.version, 'dataset': str(BRAIN_INPUT), 'run_demo': RUN_DEMO})

In [ ]:
def clone_at(url, ref, destination):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', ref], check=False)
        subprocess.run(['git', '-C', str(destination), 'checkout', ref], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', ref, url, str(destination)], check=True)

clone_at(PROJECT_URL, PROJECT_REF, PROJECT)
clone_at(FLYGIM_URL, FLYGIM_REF, FLYGIM)
print('Project HEAD:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip())
print('FlyGym HEAD:', subprocess.check_output(['git', '-C', str(FLYGIM), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
assert BRAIN_INPUT.is_dir(), f'Attach the Kaggle Dataset first: {BRAIN_INPUT}'
source = BRAIN_INPUT / 'fly-brain' if (BRAIN_INPUT / 'fly-brain').is_dir() else BRAIN_INPUT
required = ['brain_body_bridge.py', 'code/run_pytorch.py', 'code/benchmark.py', 'data/2025_Completeness_783.csv', 'data/2025_Connectivity_783.parquet', 'data/plastic_weights.pt']
missing = [item for item in required if not (source / item).is_file()]
assert not missing, f'Missing brain Dataset files: {missing}'
BRAIN.parent.mkdir(parents=True, exist_ok=True)
if BRAIN.exists():
    shutil.rmtree(BRAIN)
shutil.copytree(source, BRAIN)
size_gib = sum(path.stat().st_size for path in BRAIN.rglob('*') if path.is_file()) / 1024**3
print(f'Brain input copied to Kaggle local disk: {size_gib:.2f} GiB')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{FLYGIM}[simulation]'], check=True)
print('Project, FlyGym, FlyGym simulation dependencies installed.')

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
dry_run = WORK / 'kaggle_alpha_syn_dopamine_dry_run.json'
subprocess.run([sys.executable, str(PROJECT / 'scripts/run_alpha_syn_dopamine.py'), '--dry-run', '--output', str(dry_run)], check=True)
document = json.loads(dry_run.read_text(encoding='utf-8'))
print({'status': document['status'], 'job_count': document['job_count'], 'gpu_execution_performed': document['gpu_execution_performed']})
assert document['status'] == 'DRY_RUN_PASS' and document['job_count'] == 15
assert document['gpu_execution_performed'] is False

In [ ]:
if not RUN_DEMO:
    print('Setup complete. Demo is OFF. Set RUN_DEMO = True and rerun this cell for one 100-step rollout.')
else:
    if DEMO_OUTPUT.exists() and any(DEMO_OUTPUT.iterdir()):
        raise RuntimeError(f'Refusing to overwrite existing demo output: {DEMO_OUTPUT}')
    command = [sys.executable, str(PROJECT / 'scripts/run_neural_experiment.py'), '--brain-root', str(BRAIN), '--platform-root', str(FLYGIM), '--brain-python', sys.executable, '--annotations', str(PROJECT / 'annotations/neuron_annotations.csv'), '--age-days', '5', '--seed', '0', '--steps', str(DEMO_STEPS), '--device', 'cuda', '--output', str(DEMO_OUTPUT)]
    print('Running one non-scientific demo rollout.')
    subprocess.run(command, check=True)
    print('Demo output:', DEMO_OUTPUT)

In [ ]:
if DEMO_OUTPUT.is_dir():
    manifest = DEMO_OUTPUT / 'manifest.json'
    print('Output size MiB:', round(sum(path.stat().st_size for path in DEMO_OUTPUT.rglob('*') if path.is_file()) / 1024**2, 1))
    print('manifest exists:', manifest.is_file())
    if manifest.is_file():
        print('frame_count:', json.loads(manifest.read_text(encoding='utf-8')).get('frame_count'))
else:
    print('No demo output was created.')